# Modelo final y temporada en curso

## Qué responde este notebook

Las etapas 4 y 5 eligieron y evaluaron un modelo por resultado con temporadas ya cerradas. Este
notebook deja listos los modelos que aplica el flujo semanal (`weekly/wr/src/pipeline.py`) y los
pone a prueba con la temporada 2026, que está en curso y no se usó en ningún paso anterior:

1. Calcular las métricas de referencia del modelo elegido (validación, prueba y año por año), que
   se guardan con cada modelo.
2. Entrenar los modelos finales con 2016-2025.
3. Validarlos con las semanas de 2026 ya jugadas.
4. Guardarlos en `weekly/wr/models/` y verificar que, recargados como los carga el flujo semanal,
   predicen lo mismo.
5. Predecir la siguiente semana de 2026, todavía sin jugarse.

Este es el único notebook que escribe en `weekly/wr/models/`. En `weekly/wr/models/exploratorio/` la
etapa 4 deja solo los modelos de cuantiles de 4.5, el jerárquico de 4.7 que reutiliza 4.9 (sin
versionar) y la metadata de las alternativas que no se adoptaron.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos, pipeline

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

CONFIG_ELEGIDA = {
    "receptions": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="count:poisson", max_depth=3, n_estimators=200, learning_rate=0.08),
}

def nuevo_modelo(target):
    return XGBRegressor(**CONFIG_ELEGIDA[target], random_state=0, n_jobs=-1)

def redondear(metricas):
    return {k: round(float(v), 4) for k, v in metricas.items() if v is not None and not np.isnan(v)}

## 1. ¿Dónde va la temporada 2026?

In [2]:
stats_2026 = data.cargar_stats_semanales([2026])
wr_2026 = stats_2026[stats_2026["position"] == "WR"]
ultima_semana_jugada = int(wr_2026["week"].max())
proxima_semana = ultima_semana_jugada + 1
print(f"ultima semana jugada en 2026: {ultima_semana_jugada} | siguiente semana a predecir: {proxima_semana}")
print(f"filas de WR ya jugadas en 2026: {len(wr_2026)}")

ultima semana jugada en 2026: 4 | siguiente semana a predecir: 5
filas de WR ya jugadas en 2026: 608


## 2. Métricas de referencia del modelo elegido

Las mismas particiones de la etapa 4 (entrenamiento 2016-2021, validación 2022-2023, prueba
2024-2025) y el walk-forward de
[`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb). Se recalculan aquí porque se
guardan en la metadata de cada modelo: cada semana, el flujo semanal compara la métrica de la semana
contra estos valores.

In [3]:
wr = features.construir_tabla_modelado(range(2016, 2027))
features_arbol = features.columnas_por_modelo("arbol")
X = wr[features_arbol]
train_mask, val_mask, test_mask = experimentos.dividir_temporal(wr, range(2016, 2022), [2022, 2023], [2024, 2025])

referencias = {}
for target in modeling.TARGETS:
    y = wr[target]
    media = y[train_mask].mean()
    modelo = nuevo_modelo(target).fit(X[train_mask], y[train_mask])
    m_val = modeling.metricas_target(target, y[val_mask], modeling.clip_no_negativo(modelo.predict(X[val_mask])), media)
    m_test = modeling.metricas_target(target, y[test_mask], modeling.clip_no_negativo(modelo.predict(X[test_mask])), media)
    walk_forward = {}
    for anio, train_fold, test_fold in experimentos.folds_walk_forward(wr, [2021, 2022, 2023, 2024, 2025]):
        m = modeling.metricas_target(target, y[test_fold],
                                     modeling.clip_no_negativo(nuevo_modelo(target).fit(X[train_fold], y[train_fold]).predict(X[test_fold])),
                                     y[train_fold].mean())
        walk_forward[str(anio)] = redondear({k: m[k] for k in (modeling.METRICA_PRINCIPAL[target], "r2_oos", "d2_oos", "pendiente_calibracion") if k in m})
    cumple, detalle = experimentos.cumple_requisitos(target, m_val, {a: m for a, m in walk_forward.items()})
    referencias[target] = {"validacion": redondear(m_val), "prueba": redondear(m_test), "walk_forward": walk_forward,
                           "cumple_requisitos": bool(cumple)}

tabla_referencias = pd.DataFrame({
    target: {"metrica_principal": modeling.METRICA_PRINCIPAL[target],
             "validacion": r["validacion"][modeling.METRICA_PRINCIPAL[target]],
             "prueba": r["prueba"][modeling.METRICA_PRINCIPAL[target]],
             "r2_oos_validacion": r["validacion"]["r2_oos"], "r2_oos_prueba": r["prueba"]["r2_oos"],
             "r2_oos_minimo_por_anio": min(m["r2_oos"] for m in r["walk_forward"].values()),
             "cumple_requisitos": r["cumple_requisitos"]}
    for target, r in referencias.items()
}).T
tabla_referencias

,metrica_principal,validacion,prueba,r2_oos_validacion,r2_oos_prueba,r2_oos_minimo_por_anio,cumple_requisitos
receptions,rmse,1.9293,1.8258,0.4377,0.4585,0.4305,True
receiving_yards,rmse,29.5718,27.9143,0.3624,0.3754,0.3521,True
receiving_tds,deviance_poisson,0.6167,0.6252,0.0826,0.0877,0.0792,True


**Lectura.** Las métricas de referencia coinciden con las de [`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb)
(validación y prueba) y con el peor año de
[`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb), y los tres modelos cumplen los
requisitos. Son las cifras que se guardan con cada modelo.

## 3. Entrenar los modelos finales con 2016-2025

Con la configuración ya elegida, se entrena con todas las temporadas completas. La media de
entrenamiento de cada resultado se guarda también: es la referencia del R² (y del D²) fuera de
muestra que el flujo semanal calcula cada semana.

In [4]:
final_mask = wr["season"] <= 2025
modelos_finales, medias_finales = {}, {}
for target in modeling.TARGETS:
    modelos_finales[target] = nuevo_modelo(target).fit(X[final_mask], wr.loc[final_mask, target])
    medias_finales[target] = float(wr.loc[final_mask, target].mean())
print(f"filas de entrenamiento: {final_mask.sum()} (2016-2025)")
print("medias de entrenamiento:", {t: round(m, 4) for t, m in medias_finales.items()})

filas de entrenamiento: 23610 (2016-2025)
medias de entrenamiento: {'receptions': 2.6981, 'receiving_yards': 34.2784, 'receiving_tds': 0.2088}


## 4. Validar con 2026 ya jugado

Estas semanas no se usaron para elegir las variables ni los modelos. Se compara la métrica
principal de 2026 contra las de referencia de la sección 2.

In [5]:
mask_2026 = wr["season"] == 2026
print(f"filas de 2026 ya jugadas: {mask_2026.sum()}, semanas {sorted(int(w) for w in wr.loc[mask_2026, 'week'].unique())}")
pred_2026 = {}
filas = {}
for target in modeling.TARGETS:
    principal = modeling.METRICA_PRINCIPAL[target]
    pred_2026[target] = modeling.clip_no_negativo(modelos_finales[target].predict(X[mask_2026]))
    m = modeling.metricas_target(target, wr.loc[mask_2026, target], pred_2026[target], medias_finales[target])
    referencias[target]["temporada_2026"] = {"semanas": f"1-{ultima_semana_jugada}", **redondear(m)}
    filas[target] = {"metrica_principal": principal, "2026": m[principal],
                     "validacion_2022_2023": referencias[target]["validacion"][principal],
                     "prueba_2024_2025": referencias[target]["prueba"][principal],
                     "r2_oos_2026": m["r2_oos"], "d2_oos_2026": m.get("d2_oos", np.nan),
                     "pendiente_2026": m["pendiente_calibracion"], "sesgo_2026": m["sesgo"],
                     "sesgo_media_entrenamiento_2026": m["sesgo_media_entrenamiento"], "media_real_2026": m["media_real"]}
pd.DataFrame(filas).T.infer_objects().round(4)

filas de 2026 ya jugadas: 608, semanas [1, 2, 3, 4]


,metrica_principal,2026,validacion_2022_2023,prueba_2024_2025,r2_oos_2026,d2_oos_2026,pendiente_2026,sesgo_2026,sesgo_media_entrenamiento_2026,media_real_2026
receptions,rmse,1.8597,1.9293,1.8258,0.4624,0.4632,1.0785,-0.1526,0.2951,2.4030
receiving_yards,rmse,29.6554,29.5718,27.9143,0.3613,NaN,1.0968,-3.1483,2.4346,31.8438
receiving_tds,deviance_poisson,0.6302,0.6167,0.6252,0.0825,0.1240,0.9777,-0.0138,0.0163,0.1924


**Lectura.** En las semanas 1-4 de 2026 (608 filas), recepciones queda entre las dos referencias
(RMSE de 1.860, contra 1.929 en validación y 1.826 en prueba); yardas y touchdowns quedan un poco por
encima de las dos (RMSE de 29.66, contra 29.57 y 27.91; deviance de 0.630, contra 0.617 y 0.625). En
R² fuera de muestra los tres quedan al nivel de validación: 0.462, 0.361 y 0.0825, contra 0.438,
0.362 y 0.0826. Las pendientes de calibración quedan dentro del rango 0.9-1.1 (1.0785, 1.0968 y
0.9777).

Un punto a vigilar: en estas cuatro semanas el modelo predice por debajo de lo real (−0.15
recepciones, −3.1 yardas y −0.014 touchdowns por receptor; en yardas es 10% del promedio real).
Predecir la media de entrenamiento se habría equivocado en la dirección contraria (+2.4 yardas), así
que no es una deriva heredada del promedio. Tampoco coincide con lo visto en la prueba 2024-2025,
donde las semanas 1-4 iban al revés (+2.61 yardas, en
[`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb)). Con cuatro semanas es pronto para concluir; el flujo
semanal compara cada semana contra estas referencias.

## 5. Guardar los modelos y verificar la recarga

`experimentos.guardar_modelo` guarda el archivo del modelo (formato nativo de XGBoost) y su metadata:
variables en orden, hiperparámetros, media de entrenamiento, métrica principal, métricas de
referencia, requisitos y referencias de la política de métricas
([`docs/decisions/0004-metricas-de-seleccion.md`](../../../docs/decisions/0004-metricas-de-seleccion.md)).
Después se recargan con `modeling.cargar_modelos`, la misma función del flujo semanal, y se
comparan sus predicciones contra las de la sección 4, fila por fila.

In [6]:
cobertura_depth_2026 = wr.loc[mask_2026, "depth_team"].notna().mean()
print(f"filas de 2026 ya jugadas con depth_team: {cobertura_depth_2026:.1%}")
limitaciones_comunes = [
    "predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa semanas atipicas (ver 5.3_explicabilidad_y_casos.ipynb)",
    f"depth_team tiene dato en {cobertura_depth_2026:.1%} de las filas de 2026 ya jugadas; los faltantes son receptores que no aparecen en la alineacion de esa semana (ver 1.3_unificacion_alineaciones.ipynb)",
]
for target in modeling.TARGETS:
    limitaciones = list(limitaciones_comunes)
    if target == "receiving_tds":
        limitaciones.append("evento raro (mayoria de semanas en cero): el R2 alcanzable es bajo aun con un modelo calibrado (ver 4.1_preparacion_y_metricas.ipynb)")
    experimentos.guardar_modelo(
        modelos_finales[target], "../models", target, features_arbol, medias_finales[target], referencias[target],
        entrenado_con="2016-2025 (temporadas completas); configuracion elegida en 4.4_ajuste_hiperparametros.ipynb",
        limitaciones=limitaciones,
    )

modelos_recargados = modeling.cargar_modelos("../models")
recargadas = modeling.predecir(wr[mask_2026], modelos_recargados)
for target in modeling.TARGETS:
    identicas = np.allclose(recargadas[f"{target}_pred"].to_numpy(), pred_2026[target], atol=1e-6)
    print(f"{target}: predicciones identicas tras recargar desde disco -> {identicas}")
    assert identicas, f"el modelo recargado de {target} no coincide con el original"

filas de 2026 ya jugadas con depth_team: 95.4%
receptions: predicciones identicas tras recargar desde disco -> True
receiving_yards: predicciones identicas tras recargar desde disco -> True
receiving_tds: predicciones identicas tras recargar desde disco -> True


**Lectura.** Los tres modelos recargados desde disco predicen lo mismo que los originales, fila por
fila. La cobertura de `depth_team` en las semanas de 2026 ya jugadas (95.4%) queda anotada en la
metadata como limitación.

## 6. Predicción de la siguiente semana

Se siguen los mismos pasos que el flujo semanal: revisar que la semana no haya empezado, armar una
fila por receptor de la alineación de esa semana (`features.filas_de_la_semana`), calcular sus
variables con la historia real y aplicar los modelos recargados. Si la alineación todavía no está
completa, se avisa qué equipos faltan.

In [7]:
try:
    estado, equipos = pipeline.estado_de_la_semana(2026, proxima_semana)
except ValueError as error:
    estado, equipos = "no disponible", set()
    print(error)

if estado == "pendiente":
    filas_extra = features.filas_de_la_semana(2026, proxima_semana)
    sin_alineacion = sorted(equipos - set(filas_extra["team"]))
    print(f"semana {proxima_semana}: {len(filas_extra)} WR en la alineacion; equipos sin alineacion todavia: "
          f"{len(sin_alineacion)} de {len(equipos)} {sin_alineacion if sin_alineacion else ''}")
    tabla = features.construir_tabla_modelado(range(2016, 2027), filas_extra=filas_extra)
    semana = tabla[(tabla["season"] == 2026) & (tabla["week"] == proxima_semana)]
    print("variables con mas nulos en estas filas:")
    print(semana[features_arbol].isna().mean().sort_values(ascending=False).head(3).round(3).to_string())
    prediccion = modeling.predecir(semana, modelos_recargados).sort_values("receiving_yards_pred", ascending=False)
    display(prediccion.head(15).round(2))
else:
    print(f"semana {proxima_semana}: {estado}")

semana 5: 186 WR en la alineacion; equipos sin alineacion todavia: 0 de 30 

variables con mas nulos en estas filas:
fantasy_points_ppr_season_avg    0.086
targets_season_avg               0.086
receiving_tds_season_avg         0.086


,player_id,player_display_name,team,season,week,receptions_pred,receiving_yards_pred,receiving_tds_pred
22874,00-0039075,Puka Nacua,LA,2026,5,6.48,85.10,0.56
18191,00-0036358,CeeDee Lamb,DAL,2026,5,6.87,84.44,0.49
21761,00-0038543,Jaxon Smith-Njigba,SEA,2026,5,6.27,81.06,0.50
20327,00-0037239,Chris Olave,NO,2026,5,6.25,77.73,0.57
19841,00-0036963,Amon-Ra St. Brown,DET,2026,5,6.86,76.98,0.58
17853,00-0036322,Justin Jefferson,MIN,2026,5,5.60,76.56,0.48
22796,00-0039064,Zay Flowers,BAL,2026,5,5.50,76.42,0.39
20267,00-0037238,Drake London,ATL,2026,5,4.95,72.95,0.49
19530,00-0036900,Ja'Marr Chase,CIN,2026,5,5.87,72.44,0.53
18519,00-0036410,Tee Higgins,CIN,2026,5,5.07,71.13,0.41


La semana 5 tiene alineación completa en los 30 equipos que juegan: 186 receptores con predicción.
**Lectura.** Las variables con más faltantes son los promedios de la temporada en curso (8.6%), de
receptores que todavía no juegan en 2026. Encabezan la lista Puka Nacua, CeeDee Lamb, Jaxon
Smith-Njigba y Chris Olave, con 78 a 85 yardas esperadas y una tasa de 0.49-0.57 touchdowns.

## Cierre

Los tres modelos del flujo semanal quedan entrenados con 2016-2025 y guardados en
`weekly/wr/models/`, con su metadata completa: variables en orden, media de entrenamiento, métrica
principal, métricas de referencia (validación, prueba, año por año y 2026), requisitos y referencias
de la política de métricas. Recargados con la misma función del flujo semanal predicen idéntico.

Con la temporada 2026, que no se usó en ningún paso anterior, los tres quedan al nivel de validación
en R² fuera de muestra; en la métrica principal, recepciones queda entre las dos referencias y yardas
y touchdowns un poco por encima. Lo que hay que vigilar es una subestimación en las primeras cuatro
semanas, sobre todo en yardas (−3.1 por receptor). La semana 5, todavía sin jugarse, ya tiene predicción.

Anterior: [`5.3_explicabilidad_y_casos.ipynb`](5.3_explicabilidad_y_casos.ipynb) · Siguiente:
[`6.2_cierre.ipynb`](6.2_cierre.ipynb)